## ดึงข้อมูลจากทุก Folder มาต่อกัน โดยเลือกเฉพาะ File: transcript_backbiter.csv เท่านั้น

In [ ]:
import pandas as pd
import glob
import os

root_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/CANDOR Corpus - Conversations (no video)'

# ส่วนที่ 1: รวมไฟล์ transcript_backbiter.csv
search_pattern_1 = os.path.join(root_path, '**', 'transcript_backbiter.csv')
file_paths_1 = glob.glob(search_pattern_1, recursive=True)
all_transcripts = []

for file in file_paths_1:
    try:
        df = pd.read_csv(file)
        conv_id = os.path.basename(os.path.dirname(os.path.dirname(file)))
        df['conversation_id'] = conv_id
        all_transcripts.append(df)
    except Exception as e:
        print(f"อ่านไฟล์ transcript {file} ไม่สำเร็จ: {e}")

if all_transcripts:
    combined_transcript = pd.concat(all_transcripts, ignore_index=True)
    output_path_1 = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_backbiter.csv'
    combined_transcript.to_csv(output_path_1, index=False, encoding='utf-8-sig')
    print(f"บันทึก Transcript เรียบร้อย: {output_path_1}")

# ส่วนที่ 2: รวมไฟล์ survey.csv
search_pattern_2 = os.path.join(root_path, '**', 'survey.csv')
file_paths_2 = glob.glob(search_pattern_2, recursive=True)
all_surveys = []

for file in file_paths_2:
    try:
        df = pd.read_csv(file)
        all_surveys.append(df)
    except Exception as e:
        print(f"อ่านไฟล์ survey {file} ไม่สำเร็จ: {e}")

if all_surveys:
    combined_survey = pd.concat(all_surveys, ignore_index=True)
    output_path_2 = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_survey.csv'
    combined_survey.to_csv(output_path_2, index=False, encoding='utf-8-sig')
    print(f"บันทึก Survey เรียบร้อย: {output_path_2}")

In [ ]:
from IPython.display import display
print(" ตัวอย่างข้อมูล combined_transcript_backbiter.csv ")
display(pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_backbiter.csv").head(6))
print("\n ตัวอย่างข้อมูล combined_survey.csv ")
display(pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_survey.csv").head(6))

## Selection specifically columns: user_id, partner_id, convo_id, age, conv_leader from combined_survey.csv only

In [ ]:
combined_survey_final = combined_survey[['user_id', 'partner_id', 'convo_id', 'age', 'conv_leader']]
output_path_3 = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_survey_final.csv'
combined_survey_final.to_csv(output_path_3, index=False, encoding='utf-8-sig')
print(f"บันทึก combined_survey_final เรียบร้อย: {output_path_3}")

In [ ]:
print(" ตัวอย่างข้อมูล combined_survey_final.csv")
pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_survey_final.csv").head(6)

## Selection specifically columns: n_words, questions, speaker, delta, conversation_id from combined_transcript.csv only

In [ ]:
combined_transcript_final = combined_transcript[['n_words', 'questions', 'speaker', 'delta', 'conversation_id']]
output_path_4 = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final.csv'
combined_transcript_final.to_csv(output_path_4, index=False, encoding='utf-8-sig')
print(f"บันทึก combined_transcript_final เรียบร้อย: {output_path_4}")

In [ ]:
print(" ตัวอย่างข้อมูล combined_transcript_final.csv")
pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final.csv").head(6)

In [ ]:
import pandas as pd

# 1. โหลดข้อมูล
transcript_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final.csv'
survey_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_survey_final.csv'

transcript = pd.read_csv(transcript_path)
survey = pd.read_csv(survey_path)

print(f"โหลดข้อมูลเรียบร้อย")

# Step 1: Aggregrate columns: questions, n_words, delta group by conversation_id and speaker
speaker_level = transcript.groupby(['conversation_id', 'speaker'])[['questions', 'n_words', 'delta']].sum().reset_index()
# rename columns
speaker_level.rename(columns={
    'questions': 'total_questions',
    'n_words': 'total_words',
    'delta': 'total_speaking_time'
}, inplace=True)

# Step 2: ดึงข้อมูลจาก Survey
# 2.1 ดึงอายุ (Age) -> ดูจากแถวตัวเอง (user_id = speaker)
survey_age = survey[['convo_id', 'user_id', 'age']]
merged_df = pd.merge(speaker_level, survey_age, 
                     left_on=['conversation_id', 'speaker'], 
                     right_on=['convo_id', 'user_id'], 
                     how='inner')

# 2.2 ดึงคะแนนผู้นำ (Conv Leader) -> ดูจากแถวที่คนอื่นให้เรา (partner_id = speaker)
survey_score = survey[['convo_id', 'partner_id', 'conv_leader']]
merged_df = pd.merge(merged_df, survey_score,
                     left_on=['conversation_id', 'speaker'], 
                     right_on=['convo_id', 'partner_id'], 
                     how='inner')

# (ตอนนี้ merged_df จะมีคะแนน conv_leader ที่ "ได้รับ" จากคู่สนทนาแล้ว)

# Step 3: คำนวณ Age Diff (เหมือนเดิม)
merged_df['age_diff'] = merged_df.groupby('conversation_id')['age'].transform(lambda x: x.max() - x.min())

# Step 4: ตัดสิน Question Leader (Elder vs Younger)
leader_map = {}

for conv_id, group in merged_df.groupby('conversation_id'):
    if len(group) == 2:
        group = group.sort_values('age')
        younger = group.iloc[0]
        older = group.iloc[1]
        
        winner = 'Tie'
        
        if older['age'] > younger['age']:
            if older['total_questions'] > younger['total_questions']:
                winner = 'Elder'
            elif younger['total_questions'] > older['total_questions']:
                winner = 'Younger'
        else:
            winner = 'Same Age'
            
        leader_map[conv_id] = winner

merged_df['question_leader'] = merged_df['conversation_id'].map(leader_map)

# Step 5: จัดระเบียบและบันทึก
# ลบคอลัมน์ ID ที่ใช้ merge แล้วไม่จำเป็นต้องโชว์
cols_to_keep = ['conversation_id', 'speaker', 'age', 'conv_leader', 'age_diff', 
                'total_questions', 'total_words', 'total_speaking_time', 'question_leader']

final_df = merged_df[cols_to_keep]

# บันทึกไฟล์
output_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_speaker_summary_corrected.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n✅ แก้ไขเรียบร้อย! (คะแนน conv_leader คือคะแนนที่ได้รับจาก Partner)")
print(f"ไฟล์ใหม่อยู่ที่: {output_path}")

In [ ]:
print(" ตัวอย่างข้อมูล combined_transcript_speaker_summary_corrected.csv")
pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_speaker_summary_corrected.csv").head(6)

In [ ]:
import pandas as pd

# 1. โหลดข้อมูลไฟล์ล่าสุด (ที่แก้ Logic คะแนน Leader แล้ว)
input_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_speaker_summary_corrected.csv'
df = pd.read_csv(input_path)

print(f" โหลดข้อมูลเรียบร้อย: {len(df)} แถว")

# 2. สร้าง Dictionary เพื่อเก็บผลลัพธ์ว่าใครเป็น Actual Leader ในแต่ละคู่
actual_leader_map = {}

print(" กำลังประมวลผลหา Actual Leader (จากคะแนน Vote)...")

for conv_id, group in df.groupby('conversation_id'):
    if len(group) == 2:
        # เรียงตามอายุ (น้อย -> มาก) เพื่อระบุตัวตน
        group = group.sort_values('age')
        
        younger = group.iloc[0]
        older = group.iloc[1]
        
        winner = 'Tie' # ค่าเริ่มต้น (เสมอ)
        
        # เช็คว่าอายุต่างกันจริงไหม
        if older['age'] > younger['age']:
            # เทียบ "คะแนนความเป็นผู้นำ" (conv_leader)
            if older['conv_leader'] > younger['conv_leader']:
                winner = 'Elder'
            elif younger['conv_leader'] > older['conv_leader']:
                winner = 'Younger'
            else:
                winner = 'Tie' # คะแนนเท่ากัน
        else:
            winner = 'Same Age' # อายุเท่ากัน
            
        actual_leader_map[conv_id] = winner

# 3. ใส่ผลลัพธ์กลับเข้าไปใน DataFrame
# (คอลัมน์ใหม่: actual_leader)
df['actual_leader'] = df['conversation_id'].map(actual_leader_map)

# 4. บันทึกไฟล์ใหม่
output_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final_complete.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"\n✅ เสร็จสมบูรณ์!")
print(f"📂 บันทึกที่: {output_path}")
print("-" * 30)

# แสดงตัวอย่างข้อมูล
cols_to_show = ['conversation_id', 'age', 'conv_leader', 'question_leader', 'actual_leader']
print(df[cols_to_show].head(6))

print("-" * 30)
print("📊 สรุปผล Actual Leader (ใครได้คะแนนผู้นำเยอะกว่า?):")
print(df['actual_leader'].value_counts())

In [ ]:
print(" ตัวอย่างข้อมูล combined_transcript_final_complete.csv")
pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final_complete.csv").head(6)

In [ ]:
# check missing value
final_df = pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final_complete.csv")
print(final_df.shape)
print(final_df.isnull().sum())

In [ ]:
# 1. หา List ของ conversation_id ที่มีค่าว่าง (แม้แต่จุดเดียวในแถวใดแถวหนึ่ง)
ids_with_missing = final_df[final_df.isnull().any(axis=1)]['conversation_id'].unique()

# 2. กรองเอาเฉพาะแถวที่ conversation_id ไม่อยู่ใน List ของพวกที่มีปัญหา
final_df_cleaned = final_df[~final_df['conversation_id'].isin(ids_with_missing)]


In [ ]:
# check missing value again
print(final_df_cleaned.shape)
print(final_df_cleaned.isnull().sum())
print("---- Done!!! Do not have missing value ---")

# บันทึกไฟล์ใหม่
output_path = '/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final_complete_cleaned.csv'
final_df_cleaned.to_csv(output_path, index=False, encoding='utf-8-sig')
print("\nDone! บันทึกไฟล์สำเร็จ")

In [ ]:
# Check file ที่สมบูรณ์ กำจัด missing value
pd.read_csv("/Users/kunanon/Desktop/STATxCUYear3/INFO VISUAL/Merge Data/combined_transcript_final_complete_cleaned.csv").head(6)
